In [ ]:
# 1. Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Paket kur
!pip install ultralytics -q

In [ ]:
# 3. Dataset'i YOLO formatına dönüştür
import json
from pathlib import Path
import shutil

DATASET_ROOT = Path('/content/drive/MyDrive/dataset')
OUTPUT_ROOT  = Path('/content/yolo_dataset')  # Colab diski, hızlı
IMG_W, IMG_H = 640, 512
SPLITS = ['train', 'val', 'test']

for split in SPLITS:
    split_dir = DATASET_ROOT / split
    if not split_dir.exists():
        print(f'[SKIP] {split} bulunamadı')
        continue

    out_img = OUTPUT_ROOT / split / 'images'
    out_lbl = OUTPUT_ROOT / split / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    sequences = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    total_frames = 0
    skipped = 0

    for seq in sequences:
        json_path = seq / 'IR_label.json'
        if not json_path.exists():
            continue

        with open(json_path) as f:
            data = json.load(f)

        exist   = data['exist']
        gt_rect = data['gt_rect']
        frames  = sorted(seq.glob('*.jpg'))

        for i, frame_path in enumerate(frames):
            if i >= len(exist):
                break
            if exist[i] != 1:
                skipped += 1
                continue
            bbox = gt_rect[i]
            if bbox is None:
                skipped += 1
                continue

            x, y, w, h = bbox
            x_c = max(0.0, min(1.0, (x + w / 2) / IMG_W))
            y_c = max(0.0, min(1.0, (y + h / 2) / IMG_H))
            w_n = max(0.0, min(1.0, w / IMG_W))
            h_n = max(0.0, min(1.0, h / IMG_H))

            unique_name = f'{seq.name}_{frame_path.stem}'

            # Colab'da symlink yerine hardlink (Drive -> local disk)
            link_path = out_img / (unique_name + '.jpg')
            if not link_path.exists():
                link_path.symlink_to(frame_path.resolve())

            lbl_path = out_lbl / (unique_name + '.txt')
            with open(lbl_path, 'w') as f:
                f.write(f'0 {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n')

            total_frames += 1

    print(f'[{split}] {total_frames} frame etiketlendi, {skipped} atlandı')

# YAML
yaml_path = OUTPUT_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    f.write(f'''path: {OUTPUT_ROOT}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['uav']
''')
print('YAML hazır:', yaml_path)

In [ ]:
# 4. Eğitim
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device='cuda',
    project='/content/drive/MyDrive/runs',  # sonuçlar Drive'a kaydolur
    name='uav_detector',
    patience=10,
    save_period=5,
    val=True,
)